# Gold — Fato de Passagens

Esta tabela representa a camada analítica detalhada de passagens aéreas.

Além dos dados financeiros e operacionais, incorpora as classificações de justificativas produzidas pelo fluxo de NLP, regras, LLM e revisão humana.

In [0]:
from pyspark.sql import functions as F

df_passagens_silver = spark.table(
    "viagens_teste.silver.passagens"
)

print("Registros Silver:", df_passagens_silver.count())

### Construção da fato de passagens

Selecionamos os atributos necessários para análises financeiras, operacionais e semânticas das viagens.

In [0]:
df_fato_passagens = df_passagens_silver.select(

    F.col("id_solicitacao"),

    F.col("viajante"),

    F.col("data_solicitacao"),
    F.col("data_viagem"),

    F.col("centro_custo"),
    F.col("diretoria"),

    F.col("origem"),
    F.col("destino"),

    F.col("valor_numerico").alias("valor"),

    F.col("situacao"),
    F.col("agencia"),

    F.col("dias_antecedencia"),
    F.col("prazo"),
    F.col("trimestre"),

    F.col("justificativa_original"),
    F.col("justificativa_limpa"),

    F.col("tipo_atividade_final").alias("tipo_atividade"),
    F.col("tema_justificativa_final").alias("tema_justificativa"),

    F.col("metodo_decisao_final"),

    F.lit("Aereo").alias("tipo_despesa")
)

In [0]:
display(df_fato_passagens)

### Validação da Gold

Verificamos a quantidade de registros e a presença dos rótulos analíticos antes da persistência.

In [0]:
print(
    "Total de passagens:",
    df_fato_passagens.count()
)

print(
    "Tipo de atividade ausente:",
    df_fato_passagens
    .filter(F.col("tipo_atividade").isNull())
    .count()
)

print(
    "Tema ausente:",
    df_fato_passagens
    .filter(F.col("tema_justificativa").isNull())
    .count()
)

In [0]:
(
    df_fato_passagens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.gold.fato_passagens"
    )
)

In [0]:
display(
    spark.table(
        "viagens_teste.gold.fato_passagens"
    )
)